# 인코더(이해) vs 디코더(생성)
같은 Transformer라도 문맥을 보는 방향이 잘하는 일을 가릅니다.
* 요약·번역·챗봇은 텍스트를 생성합니다 → 생성형(디코더) 모델.
* 문의 유형 분류는 텍스트를 이해해 라벨을 답합니다 → 인코더 계열이 효율적.

|	|GPT(디코더)|	BERT(인코더)|
|---|---|---|
|방향|	단방향(왼→오)	|양방향|
|학습 목표|	다음 토큰 예측|	빈칸(마스크) 맞히기|
|잘하는 일|	생성·대화·요약|	이해·분류·검색|

인코더-디코더를 연결하면 이해+생성 둘 다 잘하는 모델이 됨. => 변환(통역 등)

### 작업별 모델 선택
판단 기준은 간단합니다 — "답이 라벨/검색결과(이해)냐, 아니면 새 문장(생성)이냐?"

|작업|	답의 형태|	적합 계열|	쇼핑몰 예시|
|---|---|---|---|
|문의 유형 분류|	라벨 1개|	BERT(인코더)|	"이 문의는 '환불'" 자동 태깅|
|유사 문서 검색·임베딩|	벡터·순위	|BERT(인코더)	|FAQ에서 비슷한 질문 찾기|
|감정/긴급도 판정|	라벨	|BERT(인코더)	|악성 리뷰 자동 분류|
|답변 생성·챗봇|	새 문장	|GPT(디코더)	|고객 문의에 답변 작성|
|요약|	새 문장	|GPT(디코더)	|긴 리뷰 3줄 요약|
|번역|	새 문장|	T5(인코더-디코더) / GPT	|해외 상품 설명 한글화|

# 양방향 vs 단방향(causal) 어텐션

In [ ]:
import numpy as np

# softmax: 점수들을 합이 1인 '주목 비율'로 바꿔주는 함수(어텐션의 핵심 연산)
def softmax(x):
    e = np.exp(x - x.max(-1, keepdims=True))   # 최댓값 빼기 = 큰 수 지수폭주 방지(안정화)
    return e / e.sum(-1, keepdims=True)         # 각 행의 합이 1이 되도록 정규화

n = 4                                           # 단어 4개짜리 짧은 문장이라 가정
scores = np.zeros((n, n))                       # 단어 간 유사도가 모두 같다고 단순화(0점)

# 디코더(GPT)용 causal mask: 미래(오른쪽) 자리를 -무한대로 막아 주목 비율을 0으로 만든다
# np.triu(..., k=1) = 대각선 위쪽(=미래) 영역만 선택
causal = np.triu(np.full((n, n), -np.inf), k=1)

print('인코더(BERT, 양방향):')
print(softmax(scores).round(2))                 # 모든 단어가 앞뒤 모두에 고르게 주목
print('\n디코더(GPT, 단방향):')
print(softmax(scores + causal).round(2))        # i번째 단어는 자기 이하(j<=i)에만 주목

인코더(BERT, 양방향):
[[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]]

디코더(GPT, 단방향):
[[1.   0.   0.   0.  ]
 [0.5  0.5  0.   0.  ]
 [0.33 0.33 0.33 0.  ]
 [0.25 0.25 0.25 0.25]]


관찰 포인트
- BERT(위) 표는 모든 칸이 0.25 — 4개 단어에 골고루 주목(= 앞뒤 다 봄).
- GPT(아래) 표는 왼쪽 아래 삼각형만 값이 있고 오른쪽 위(미래)는 0 — 자기 앞만 봄.
- 이 "미래 0" 규칙 하나가 GPT를 다음 단어 예측(생성) 모델로 만듭니다.



# 빈칸 채우기(BERT) vs 이어쓰기(GPT)
인코더(BERT)의 fill-mask와 디코더(GPT)의 text-generation을 직접 돌려, 두 계열의 차이를 이해합니다.

### 0.준비
pipeline: 토크나이즈 → 모델 추론 → 후처리를 한 줄로 묶어주는 고수준 API

In [ ]:
#!pip install -q transformers torch
from transformers import pipeline

# 1. 빈칸 채우기(BERT)
BERT는 양옆 문맥을 보고 [MASK] 자리에 들어갈 단어를 확률 순으로 제안합니다(이해형).

In [ ]:
# 'fill-mask'(빈칸 채우기) 작업 + 한국어 BERT 지정(처음이면 모델을 자동 내려받음)
fill = pipeline('fill-mask', model='klue/bert-base')
mask = fill.tokenizer.mask_token          # 이 모델의 빈칸 토큰 문자열(보통 '[MASK]')
# 빈칸 자리에 들어갈 후보를 확률 높은 순으로 받아, 상위 3개만([:3])
for pred in fill(f'이 티셔츠는 사이즈가 약간 {mask}.')[:3]:
    print(round(pred['score'], 3), pred['token_str'], '->', pred['sequence'])
    #     score=확률,            token_str=채운 단어,   sequence=완성 문장

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

0.268 크다 -> 이 티셔츠는 사이즈가 약간 크다.
0.221 [UNK] -> 이 티셔츠는 사이즈가 약간.
0.195 다릅니다 -> 이 티셔츠는 사이즈가 약간 다릅니다.


In [ ]:
exclude_tokens = ['[UNK]', '[PAD]', '[CLS]', '[SEP]', '[MASK]', '.', ',', '!', '?']

for pred in fill(f'이 티셔츠는 사이즈가 약간 {mask}.'):
    if pred['token_str'].strip() not in exclude_tokens:
        print(round(pred['score'], 3), pred['token_str'], '->', pred['sequence'])

0.268 크다 -> 이 티셔츠는 사이즈가 약간 크다.
0.195 다릅니다 -> 이 티셔츠는 사이즈가 약간 다릅니다.
0.1 달라요 -> 이 티셔츠는 사이즈가 약간 달라요.


# 2. 이어쓰기(GPT)

GPT는 주어진 시작 문구에 이어서 문장을 생성합니다(생성형).



In [ ]:
# 'text-generation'(이어쓰기) 작업 + 한국어 GPT-2 지정
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel, pipeline

model_id = 'skt/kogpt2-base-v2'

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    model_id,
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>'
)

model = GPT2LMHeadModel.from_pretrained(model_id)

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

out = gen(
    "고객님, 주문하신 상품은",
    max_new_tokens=100,    # 새로 생성할 최대 토큰 수(생성 길이 제한)
    do_sample=True,       # 매번 같은 답 대신 확률적으로 샘플링(다양성 부여)
    temperature=0.8,      # 샘플링 무작위성: 낮으면 보수적·높으면 과감(0.8=적당)
    top_p=0.9,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)

text = out[0]["generated_text"]

print(text)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


고객님, 주문하신 상품은즉시 배송하는 서비스를 선보일 예정입니다.
롯데닷컴 관계자는 "추석을 맞아 추석 명절 선물 수요가 증가할 것으로 예상돼 배송 서비스의 질을 더욱 높일 수 있도록 이달부터 '홈플러스 온라인 전용 배송 서비스'를 시작했다"며 "앞으로도 고객이 원하는 상품을 빠르고 편리하게 받아볼 수 있는 쇼핑 환경을 지속해서 만들어 갈 것"이라고 말했습니다. 금융당국이 지난해 주택담보대출 급증으로 인한 가계부채 증가세가 둔화되자 대책 마련에 나섰다.
이에 따라 은행권이 고정


# huggingface transformer pipeline
"작업 종류"를 지정하면 한 줄로 처리해주는 라이브러리

|task	|용도	|모델 예|
|---|---|---|
|fill-mask|	빈칸 채우기(인코더)|	klue/bert-base|
|text-generation|	이어쓰기(디코더)|	skt/kogpt2-base-v2|
|text-classification|	분류|	파인튜닝한 분류기|
|zero-shot-classification|	학습 없이 분류|	joeddav/xlm-roberta-large-xnli|
|feature-extraction|	임베딩 추출|	임베딩 모델|


### huggingface 한국어 모델 제공
klue/bert-base — KLUE 벤치마크용 한국어 BERT. 이해·분류 실습의 기본.  
skt/kogpt2-base-v2 — 한국어 GPT-2. 생성 데모용(작아서 품질은 제한적).  
한국어는 영어 모델로는 토큰화가 비효율적이라, 한국어 전용/다국어 모델을 고르는 게 좋습니다.